# NAFNet Deblurring â€” GoPro Training Notebook
**Upload these files to Kaggle before running:**
`model.py`, `losses.py`, `dataset.py`, `train.py`, `inference.py`, `evaluate.py`, `utils.py`

**Dataset:** [gopro-image-deblurring-dataset](https://www.kaggle.com/datasets/adwythdarsanr/gopro-image-deblurring-dataset) â€” add as input

**Expected results after ~150 epochs (T4 GPU, ~8â€“12 hours):**
- PSNR â‰¥ 33 dB
- SSIM â‰¥ 0.95
- LPIPS â‰¤ 0.07

In [ ]:
# â”€â”€ Cell 1: Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install lpips timm --quiet
print('Dependencies installed.')

In [ ]:
# ── Cell 2: Verify GPU + dataset ─────────────────────────────────────────────
import os, torch, subprocess
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Kaggle mounts datasets at /kaggle/input/<slug>  OR  /kaggle/input/datasets/<user>/<slug>
_CANDIDATES = [
    "/kaggle/input/gopro-image-deblurring-dataset",
    "/kaggle/input/datasets/adwythdarsanr/gopro-image-deblurring-dataset",
]
DATA_ROOT = next((c for c in _CANDIDATES if Path(c).exists()), _CANDIDATES[0])
print("")
print("Dataset root :", DATA_ROOT, " exists=", Path(DATA_ROOT).exists())

result = subprocess.run(["find", DATA_ROOT, "-name", "*.png", "-type", "f"],
                        capture_output=True, text=True)
n_files = len([l for l in result.stdout.strip().splitlines() if l])
print("PNG files found :", n_files)


In [ ]:
# â”€â”€ Cell 3: Explore dataset structure â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from pathlib import Path

root = Path(DATA_ROOT)
for p in sorted(root.iterdir())[:5]:
    print(p.name)
    for sub in sorted(p.iterdir())[:3]:
        print(f'  {sub.name}', list(sub.iterdir())[:2] if sub.is_dir() else '')

In [ ]:
# â”€â”€ Cell 4: Validate dataset loading â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, '/kaggle/working')   # where you uploaded the .py files

from dataset import GoproDataset
import matplotlib.pyplot as plt

train_ds = GoproDataset(DATA_ROOT, split='train', patch_size=256)
test_ds  = GoproDataset(DATA_ROOT, split='test',  patch_size=256)
print(f'Train: {len(train_ds)} pairs | Test: {len(test_ds)} pairs')

blur, sharp = train_ds[0]
print(f'Blur shape: {blur.shape}  Sharp shape: {sharp.shape}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(blur.permute(1, 2, 0).numpy())
axes[0].set_title('Blurry')
axes[1].imshow(sharp.permute(1, 2, 0).numpy())
axes[1].set_title('Sharp (GT)')
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ Cell 5: Model summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from model import NAFNetDeblur

model = NAFNetDeblur(width=64, middle_blk_num=12,
                     enc_blk_nums=[2,2,4,8], dec_blk_nums=[2,2,2,2])
n = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {n:.1f}M')

# Quick forward pass
import torch
x = torch.randn(1, 3, 256, 256)
model.eval()
with torch.no_grad():
    y = model(x)
print(f'Input: {x.shape}  Output: {y.shape}')

In [ ]:
# â”€â”€ Cell 6: Loss sanity check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from losses import CombinedLoss
import torch

crit = CombinedLoss()
pred   = torch.rand(2, 3, 256, 256)
target = torch.rand(2, 3, 256, 256)
total, breakdown = crit(pred, target)
print(f'Total loss: {total.item():.4f}')
for k, v in breakdown.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── Cell 7: Start training ────────────────────────────────────────────────
# Adjust batch_size / width if you get OOM errors.
# width=32 uses ~17M params and is safe at batch_size=8.
# width=64 uses ~67M params — use batch_size=4.

from train import train, CFG

CFG.update({
    "data_root":  DATA_ROOT,
    "out_dir":    "/kaggle/working/deblur_out",
    "width":      32,
    "batch_size": 16,
    "epochs":     50,
    "loss_type":  "charbonnier",
    "use_ssim":   False,
    "use_perc":   False,
    "patience":   20,
    "resume":     None,
})

train(CFG)


In [ ]:
# â”€â”€ Cell 8: Plot training curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, matplotlib.pyplot as plt

with open('/kaggle/working/deblur_out/history.json') as f:
    hist = json.load(f)

epochs  = [r['epoch']    for r in hist]
t_psnr  = [r['train_psnr'] for r in hist]
e_psnr  = [r['ema_psnr']   for r in hist]
e_ssim  = [r['ema_ssim']   for r in hist]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(epochs, t_psnr, label='Train PSNR')
ax1.plot(epochs, e_psnr, label='EMA PSNR', linewidth=2)
ax1.axhline(33, color='r', linestyle='--', label='Target 33 dB')
ax1.set(xlabel='Epoch', ylabel='PSNR (dB)', title='PSNR')
ax1.legend()

ax2.plot(epochs, e_ssim, color='green', linewidth=2)
ax2.axhline(0.95, color='r', linestyle='--', label='Target 0.95')
ax2.set(xlabel='Epoch', ylabel='SSIM', title='EMA SSIM')
ax2.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=120)
plt.show()

In [ ]:
# â”€â”€ Cell 9: Full evaluation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from evaluate import evaluate

results = evaluate(
    ckpt_path = '/kaggle/working/deblur_out/checkpoints/best.pth',
    data_root = DATA_ROOT,
    save_imgs = True,
    out_dir   = '/kaggle/working/eval_results',
)
print(results)

In [ ]:
# â”€â”€ Cell 10: Visual inspection of results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

samples = sorted(Path('/kaggle/working/eval_results').glob('sample_*.png'))[:4]
fig, axes = plt.subplots(len(samples), 1, figsize=(18, 5 * len(samples)))

if len(samples) == 1:
    axes = [axes]

for ax, sp in zip(axes, samples):
    img = Image.open(sp)
    ax.imshow(img)
    ax.set_title(sp.stem)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/visual_results.png', dpi=100)
plt.show()

In [ ]:
# â”€â”€ Cell 11: Inference on a single custom image â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from inference import load_model, restore_image, save_side_by_side, read_image
from utils import tensor_to_uint8
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = load_model('/kaggle/working/deblur_out/checkpoints/best.pth', device)

# Replace with any blurry image path
TEST_IMG = '/kaggle/input/gopro-image-deblurring-dataset/test/.../.../blur/00001.png'

img_t    = read_image(TEST_IMG)
restored = restore_image(model, img_t, device, tile_size=512, overlap=32)

orig_np = tensor_to_uint8(img_t[0])
rest_np = tensor_to_uint8(restored[0])
save_side_by_side(orig_np, rest_np, '/kaggle/working/custom_result.png')

from PIL import Image
Image.open('/kaggle/working/custom_result.png')

In [ ]:
# â”€â”€ Cell 12: List output files for download â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os
for root, dirs, files in os.walk('/kaggle/working/deblur_out'):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1e6
        print(f'{size:6.1f} MB  {full}')